# 🎬 MovieLens 25M — Exploratory Data Analysis
## Prepared for Neural Collaborative Filtering (NCF / NeuMF) Training

---

**Dataset:** MovieLens 25M  
**Files:** `ratings.csv`, `movies.csv`, `tags.csv`, `links.csv`, `genome-scores.csv`, `genome-tags.csv`  
**Objective:** Understand the data, identify preprocessing requirements, and validate assumptions before building the NCF training pipeline.

### EDA Sections
1. [Environment Setup](#1)
2. [Dataset Ingestion & Schema Validation](#2)
3. [Data Quality Checks](#3)
4. [Rating Distribution Analysis](#4)
5. [User Activity Analysis](#5)
6. [Movie Popularity Analysis](#6)
7. [Temporal Analysis](#7)
8. [Genre Analysis](#8)
9. [Interaction Matrix Sparsity](#9)
10. [Tag & Genome Data Exploration](#10)
11. [NCF Readiness Checks](#11)
12. [Key Findings & Next Steps](#12)

---
<a id='1'></a>
## 1. Environment Setup

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
import warnings
from pathlib import Path
from itertools import combinations

# ── Data Manipulation ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)

# ── Plotting theme ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#3a3a5c',
    'axes.labelcolor': '#e0e0ff',
    'axes.titlecolor': '#c0b3ff',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.color': '#a0a0cc',
    'ytick.color': '#a0a0cc',
    'grid.color': '#2a2a4a',
    'grid.linewidth': 0.6,
    'text.color': '#e0e0ff',
    'legend.facecolor': '#1a1a2e',
    'legend.edgecolor': '#3a3a5c',
    'legend.labelcolor': '#e0e0ff',
    'font.family': 'DejaVu Sans',
})

PALETTE = ['#7c6fff', '#ff6b9d', '#43e0ad', '#ffb347', '#54c5f8', '#e87461']
ACCENT  = '#7c6fff'
ACCENT2 = '#43e0ad'

# ── Data path ─────────────────────────────────────────────────────────────────
# Adjust DATA_DIR if running from a different location
DATA_DIR = Path('../Dataset/ml-25m')

assert DATA_DIR.exists(), f'Dataset directory not found: {DATA_DIR.resolve()}'
print('✅ Environment ready')
print(f'📂 Dataset path : {DATA_DIR.resolve()}')

---
<a id='2'></a>
## 2. Dataset Ingestion & Schema Validation

In [ ]:
# ── 2.1  Load all six files ───────────────────────────────────────────────────
print('⏳ Loading ratings.csv (~678 MB) — this may take ~30 s …')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv',
                      dtype={'userId': 'int32', 'movieId': 'int32',
                             'rating': 'float32', 'timestamp': 'int64'})
print(f'   ratings      : {len(ratings):>12,} rows')

print('⏳ Loading movies.csv …')
movies = pd.read_csv(DATA_DIR / 'movies.csv')
print(f'   movies       : {len(movies):>12,} rows')

print('⏳ Loading tags.csv …')
tags = pd.read_csv(DATA_DIR / 'tags.csv',
                   dtype={'userId': 'int32', 'movieId': 'int32', 'timestamp': 'int64'})
print(f'   tags         : {len(tags):>12,} rows')

print('⏳ Loading links.csv …')
links = pd.read_csv(DATA_DIR / 'links.csv',
                    dtype={'movieId': 'int32'})
print(f'   links        : {len(links):>12,} rows')

print('⏳ Loading genome-tags.csv …')
genome_tags = pd.read_csv(DATA_DIR / 'genome-tags.csv')
print(f'   genome_tags  : {len(genome_tags):>12,} rows')

print('⏳ Loading genome-scores.csv (~435 MB) — this may take ~45 s …')
genome_scores = pd.read_csv(DATA_DIR / 'genome-scores.csv',
                             dtype={'movieId': 'int32', 'tagId': 'int32',
                                    'relevance': 'float32'})
print(f'   genome_scores: {len(genome_scores):>12,} rows')
print('\n✅ All files loaded successfully.')

In [ ]:
# ── 2.2  Schema validation ────────────────────────────────────────────────────
def show_schema(name, df):
    print(f'\n── {name} ──────────────────────────────')
    print(df.dtypes.to_string())
    print(f'Shape : {df.shape}')
    display(df.head(3))

for name, df in [('ratings', ratings), ('movies', movies),
                 ('tags', tags), ('links', links),
                 ('genome_tags', genome_tags)]:
    show_schema(name, df)

In [ ]:
# ── 2.3  High-level dataset summary ──────────────────────────────────────────
summary = {
    'Total Ratings'            : len(ratings),
    'Unique Users'             : ratings['userId'].nunique(),
    'Unique Movies (ratings)'  : ratings['movieId'].nunique(),
    'Unique Movies (metadata)' : movies['movieId'].nunique(),
    'Unique Tag Strings'       : tags['tag'].nunique(),
    'Genome Movies'            : genome_scores['movieId'].nunique(),
    'Genome Tags'              : genome_tags['tagId'].nunique(),
    'Rating Scale Min'         : ratings['rating'].min(),
    'Rating Scale Max'         : ratings['rating'].max(),
    'Date Range Start'         : pd.to_datetime(ratings['timestamp'].min(), unit='s').date(),
    'Date Range End'           : pd.to_datetime(ratings['timestamp'].max(), unit='s').date(),
}

summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
print('\n📊 Dataset High-Level Summary')
print('=' * 45)
print(summary_df.to_string(index=False))

---
<a id='3'></a>
## 3. Data Quality Checks

In [ ]:
# ── 3.1  Missing values ───────────────────────────────────────────────────────
print('Missing values per file:\n')
for name, df in [('ratings', ratings), ('movies', movies),
                 ('tags', tags), ('links', links)]:
    mv      = df.isnull().sum()
    total_mv = mv.sum()
    pct      = (mv / len(df) * 100).round(4)
    print(f'  {name:<15}  total missing = {total_mv}')
    if total_mv > 0:
        print(mv[mv > 0].to_string())
        print(pct[pct > 0].to_string())
    print()

In [ ]:
# ── 3.2  Duplicate rows ───────────────────────────────────────────────────────
print('Duplicate rows per file:\n')
for name, df in [('ratings', ratings), ('movies', movies),
                 ('tags', tags), ('links', links)]:
    dups = df.duplicated().sum()
    print(f'  {name:<15}  duplicates = {dups}')

dup_interactions = ratings.duplicated(subset=['userId', 'movieId']).sum()
print(f'\n  ratings — duplicate (userId, movieId) pairs = {dup_interactions:,}')
print('  (Multiple rows mean a user re-rated the same movie at different times.)')

In [ ]:
# ── 3.3  Rating value sanity check ───────────────────────────────────────────
valid_ratings = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
invalid_mask  = ~ratings['rating'].isin(valid_ratings)
print(f'Ratings outside 0.5–5.0 half-star scale : {invalid_mask.sum():,}')

print('\nRating value counts:')
print(ratings['rating'].value_counts().sort_index().to_string())

In [ ]:
# ── 3.4  Timestamp sanity ─────────────────────────────────────────────────────
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')

print(f'Earliest rating : {ratings["datetime"].min()}')
print(f'Latest   rating : {ratings["datetime"].max()}')
print(f'\nRatings before 1995 : {(ratings["datetime"].dt.year < 1995).sum()}')
print(f'Ratings after  2020 : {(ratings["datetime"].dt.year > 2020).sum()}')

In [ ]:
# ── 3.5  Movie metadata completeness ─────────────────────────────────────────
rated_movie_ids = set(ratings['movieId'].unique())
meta_movie_ids  = set(movies['movieId'].unique())

rated_not_meta  = rated_movie_ids - meta_movie_ids
meta_not_rated  = meta_movie_ids  - rated_movie_ids

print(f'Movies rated but missing from movies.csv : {len(rated_not_meta):,}')
print(f'Movies in movies.csv but never rated     : {len(meta_not_rated):,}')

no_genre = movies[movies['genres'] == '(no genres listed)']
print(f'\nMovies with no genre listed              : {len(no_genre):,}')
print(f'  ({len(no_genre) / len(movies) * 100:.2f}% of total movies)')

---
<a id='4'></a>
## 4. Rating Distribution Analysis

In [ ]:
# ── 4.1  Overall rating distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Rating Distribution — MovieLens 25M', fontsize=15, fontweight='bold')

rating_counts = ratings['rating'].value_counts().sort_index()

ax = axes[0]
bars = ax.bar(rating_counts.index.astype(str), rating_counts.values,
              color=PALETTE[0], edgecolor='#5a4adf', linewidth=0.6, width=0.6)
ax.set_title('Rating Value Frequencies')
ax.set_xlabel('Rating')
ax.set_ylabel('Number of Ratings')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.grid(axis='y', alpha=0.5)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 50_000,
            f'{h/1e6:.2f}M', ha='center', va='bottom', fontsize=7.5, color='#e0e0ff')

ax = axes[1]
ax.ecdf(ratings['rating'], color=ACCENT)
ax.set_title('Cumulative Distribution of Ratings')
ax.set_xlabel('Rating Value')
ax.set_ylabel('CDF')
ax.axvline(3.5, color=PALETTE[1], linestyle='--', linewidth=1.2, label='3.5 threshold')
ax.axvline(4.0, color=PALETTE[2], linestyle='--', linewidth=1.2, label='4.0 threshold')
ax.legend(fontsize=9)
ax.grid(alpha=0.5)

plt.tight_layout()
plt.show()

print('\nRating Summary Statistics:')
print(ratings['rating'].describe().to_string())

In [ ]:
# ── 4.2  Implicit feedback threshold analysis ─────────────────────────────────
# For NCF we convert explicit ratings → implicit binary feedback.
# Explore the positive/negative split at various thresholds.

thresholds = [0.5, 1.0, 2.0, 3.0, 3.5, 4.0]
results = []
for t in thresholds:
    pos = (ratings['rating'] >= t).sum()
    neg = (ratings['rating'] <  t).sum()
    results.append({
        'Threshold'             : t,
        'Positive Interactions' : pos,
        'Negative Interactions' : neg,
        'Positive %'            : round(pos / len(ratings) * 100, 2),
        'Negative %'            : round(neg / len(ratings) * 100, 2),
    })

thresh_df = pd.DataFrame(results)
print('Implicit Feedback — Positive/Negative Split by Rating Threshold:')
print(thresh_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(thresholds))
ax.bar(x,                    thresh_df['Positive %'], label='Positive', color=ACCENT,     width=0.4)
ax.bar([i+0.4 for i in x],  thresh_df['Negative %'], label='Negative', color=PALETTE[1], width=0.4)
ax.set_xticks([i+0.2 for i in x])
ax.set_xticklabels([f'>={t}' for t in thresholds])
ax.set_title('Positive vs Negative Split per Rating Threshold')
ax.set_xlabel('Rating Threshold')
ax.set_ylabel('% of All Interactions')
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print('\n💡 NCF paper treats ALL observed interactions as implicit positives.')
print('   A practical threshold of >= 3.5 or >= 4.0 can reduce noise.')

---
<a id='5'></a>
## 5. User Activity Analysis

In [ ]:
# ── 5.1  Ratings per user distribution ───────────────────────────────────────
user_counts = ratings.groupby('userId').size()

print('Ratings-per-user Summary:')
print(user_counts.describe().to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('User Activity — Ratings per User', fontsize=14, fontweight='bold')

ax = axes[0]
ax.hist(user_counts, bins=100, color=ACCENT, edgecolor='none', alpha=0.85)
ax.set_title('Distribution (Linear Scale)')
ax.set_xlabel('Number of Ratings')
ax.set_ylabel('Number of Users')
ax.axvline(user_counts.median(), color=PALETTE[1], linestyle='--',
           linewidth=1.5, label=f'Median = {int(user_counts.median())}')
ax.axvline(user_counts.mean(), color=PALETTE[2], linestyle='--',
           linewidth=1.5, label=f'Mean = {user_counts.mean():.0f}')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.4)

ax = axes[1]
ax.hist(user_counts, bins=200, color=PALETTE[3], edgecolor='none', alpha=0.85)
ax.set_title('Distribution (Log-Log — Long Tail)')
ax.set_xlabel('Number of Ratings')
ax.set_ylabel('Number of Users')
ax.set_yscale('log')
ax.set_xscale('log')
ax.grid(alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2  User activity percentiles and cold-start thresholds ──────────────────
percentiles  = [10, 20, 25, 50, 75, 80, 90, 95, 99]
pctile_vals  = np.percentile(user_counts, percentiles)

pctile_df = pd.DataFrame({'Percentile (%)': percentiles,
                           'Ratings per User': pctile_vals.astype(int)})
print('User Activity Percentiles:')
print(pctile_df.to_string(index=False))

print('\nUsers with fewer than N ratings (filtering thresholds):')
for n in [5, 10, 20, 50]:
    below = (user_counts < n).sum()
    pct   = below / len(user_counts) * 100
    print(f'  < {n:3d} ratings : {below:>7,} users  ({pct:.2f}%)')

print('\n💡 Dataset README states all users have >= 20 ratings — confirm above.')

In [ ]:
# ── 5.3  Leave-One-Out eligibility ────────────────────────────────────────────
loo_eligible   = (user_counts >= 2).sum()
loo_ineligible = (user_counts <  2).sum()

print(f'Users eligible for LOO evaluation (>= 2 ratings) : {loo_eligible:>8,}')
print(f'Users ineligible for LOO evaluation (< 2 ratings): {loo_ineligible:>8,}')
print(f'\nLOO eligibility rate : {loo_eligible / len(user_counts) * 100:.4f}%')

In [ ]:
# ── 5.4  Average rating per user ──────────────────────────────────────────────
user_avg_rating = ratings.groupby('userId')['rating'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(user_avg_rating, bins=60, color=PALETTE[4], edgecolor='none', alpha=0.85)
ax.set_title('Distribution of Per-User Average Rating')
ax.set_xlabel('Average Rating')
ax.set_ylabel('Number of Users')
ax.axvline(user_avg_rating.mean(), color=PALETTE[1], linestyle='--',
           linewidth=1.5, label=f'Grand mean = {user_avg_rating.mean():.2f}')
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print('\nPer-User Average Rating Summary:')
print(user_avg_rating.describe().to_string())

---
<a id='6'></a>
## 6. Movie Popularity Analysis

In [ ]:
# ── 6.1  Ratings per movie distribution ──────────────────────────────────────
movie_counts = ratings.groupby('movieId').size()

print('Ratings-per-movie Summary:')
print(movie_counts.describe().to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Movie Popularity — Ratings per Movie', fontsize=14, fontweight='bold')

ax = axes[0]
ax.hist(movie_counts, bins=100, color=PALETTE[1], edgecolor='none', alpha=0.85)
ax.set_title('Distribution (Linear Scale)')
ax.set_xlabel('Number of Ratings')
ax.set_ylabel('Number of Movies')
ax.axvline(movie_counts.median(), color=ACCENT, linestyle='--',
           linewidth=1.5, label=f'Median = {int(movie_counts.median())}')
ax.legend()
ax.grid(axis='y', alpha=0.4)

ax = axes[1]
ax.hist(movie_counts, bins=200, color=PALETTE[2], edgecolor='none', alpha=0.85)
ax.set_title('Distribution (Log-Log — Long Tail)')
ax.set_xlabel('Number of Ratings')
ax.set_ylabel('Number of Movies')
ax.set_yscale('log')
ax.set_xscale('log')
ax.grid(alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── 6.2  Top-20 most popular movies ──────────────────────────────────────────
top_movies = (
    movie_counts
    .nlargest(20)
    .reset_index()
    .merge(movies[['movieId', 'title']], on='movieId', how='left')
)
# Normalise column name for pandas >=2.0
count_col = [c for c in top_movies.columns if c not in ('movieId', 'title')][0]
top_movies = top_movies.rename(columns={count_col: 'count'})

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_movies['title'][::-1], top_movies['count'][::-1],
        color=PALETTE[0], edgecolor='none', alpha=0.9)
ax.set_title('Top 20 Most Rated Movies')
ax.set_xlabel('Number of Ratings')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.show()

print('\nTop 20 Most Rated Movies:')
print(top_movies[['title', 'count']].to_string(index=False))

In [ ]:
# ── 6.3  Long-tail coverage (Pareto analysis) ─────────────────────────────────
total_ratings  = movie_counts.sum()
sorted_counts  = movie_counts.sort_values(ascending=False)
cumulative_pct = sorted_counts.cumsum() / total_ratings

print('How many movies account for X% of all ratings?')
for target in [50, 80, 90, 95]:
    n_movies   = (cumulative_pct <= target / 100).sum()
    pct_movies = n_movies / len(movie_counts) * 100
    print(f'  {target}% of ratings → top {n_movies:,} movies ({pct_movies:.2f}% of catalog)')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.arange(1, len(sorted_counts)+1),
        cumulative_pct.values * 100,
        color=ACCENT, linewidth=2)
ax.axhline(80, color=PALETTE[1], linestyle='--', linewidth=1, label='80%')
ax.axhline(50, color=PALETTE[2], linestyle='--', linewidth=1, label='50%')
ax.set_title('Cumulative Rating Coverage (Long-Tail)')
ax.set_xlabel('Movie Rank (sorted by popularity)')
ax.set_ylabel('Cumulative % of Ratings')
ax.legend()
ax.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.4  Popularity vs average rating scatter ─────────────────────────────────
movie_avg = ratings.groupby('movieId').agg(
    mean_rating=('rating', 'mean'),
    n_ratings=('rating', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(movie_avg['n_ratings'], movie_avg['mean_rating'],
                alpha=0.25, s=4, c=movie_avg['mean_rating'],
                cmap='plasma', linewidths=0)
plt.colorbar(sc, ax=ax, label='Mean Rating')
ax.set_title('Movie Popularity vs Average Rating')
ax.set_xlabel('Number of Ratings (log scale)')
ax.set_ylabel('Average Rating')
ax.set_xscale('log')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('💡 Popular movies converge toward global mean (regression to mean).')
print('   Sparse movies show high rating variance — noise at the catalog edges.')

---
<a id='7'></a>
## 7. Temporal Analysis

In [ ]:
# ── 7.1  Monthly rating volume ────────────────────────────────────────────────
ratings['year_month'] = ratings['datetime'].dt.to_period('M')
monthly = ratings.groupby('year_month').size().reset_index(name='count')
monthly['year_month_dt'] = monthly['year_month'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(monthly['year_month_dt'], monthly['count'], alpha=0.3, color=ACCENT)
ax.plot(monthly['year_month_dt'], monthly['count'], color=ACCENT, linewidth=1.5)
ax.set_title('Monthly Rating Volume (1995–2019)')
ax.set_xlabel('Date')
ax.set_ylabel('Ratings per Month')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
ax.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.2  Annual rating volume and mean rating trend ───────────────────────────
ratings['year'] = ratings['datetime'].dt.year
annual = ratings.groupby('year').agg(
    n_ratings=('rating', 'count'),
    mean_rating=('rating', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Annual Rating Trends', fontsize=14, fontweight='bold')

ax = axes[0]
ax.bar(annual['year'], annual['n_ratings'] / 1e6,
       color=PALETTE[3], edgecolor='none', alpha=0.85)
ax.set_title('Annual Rating Volume')
ax.set_xlabel('Year')
ax.set_ylabel('Ratings (M)')
ax.grid(axis='y', alpha=0.4)

ax = axes[1]
ax.plot(annual['year'], annual['mean_rating'],
        color=PALETTE[4], linewidth=2, marker='o', markersize=5)
ax.set_title('Annual Mean Rating')
ax.set_xlabel('Year')
ax.set_ylabel('Average Rating')
ax.set_ylim(0, 5)
ax.grid(alpha=0.4)

plt.tight_layout()
plt.show()

print(annual.to_string(index=False))

In [ ]:
# ── 7.3  Day-of-week and hour-of-day patterns ─────────────────────────────────
ratings['dayofweek'] = ratings['datetime'].dt.day_name()
ratings['hour']      = ratings['datetime'].dt.hour

day_order   = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_counts  = ratings['dayofweek'].value_counts().reindex(day_order)
hour_counts = ratings['hour'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Rating Activity Patterns', fontsize=14, fontweight='bold')

ax = axes[0]
ax.bar(dow_counts.index, dow_counts.values / 1e6,
       color=PALETTE[2], alpha=0.85)
ax.set_title('Ratings by Day of Week')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Ratings (M)')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.4)

ax = axes[1]
ax.bar(hour_counts.index, hour_counts.values / 1e6,
       color=PALETTE[5], alpha=0.85)
ax.set_title('Ratings by Hour of Day (UTC)')
ax.set_xlabel('Hour (UTC)')
ax.set_ylabel('Ratings (M)')
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.show()

---
<a id='8'></a>
## 8. Genre Analysis

In [ ]:
# ── 8.1  Genre frequency in the movie catalog ─────────────────────────────────
genre_exploded = movies.assign(
    genre=movies['genres'].str.split('|')
).explode('genre')
genre_exploded = genre_exploded[genre_exploded['genre'] != '(no genres listed)']

genre_counts = genre_exploded['genre'].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(genre_counts.index[::-1], genre_counts.values[::-1],
        color=PALETTE[0], edgecolor='none', alpha=0.9)
ax.set_title('Genre Frequency in Movie Catalog')
ax.set_xlabel('Number of Movies')
ax.grid(axis='x', alpha=0.4)
for bar in ax.patches:
    w = bar.get_width()
    ax.text(w + 30, bar.get_y() + bar.get_height() / 2,
            f'{int(w):,}', va='center', fontsize=8, color='#e0e0ff')
plt.tight_layout()
plt.show()

print('\nGenre Counts:')
print(genre_counts.to_string())

In [ ]:
# ── 8.2  Number of genres per movie ───────────────────────────────────────────
movies['genre_count'] = movies['genres'].apply(
    lambda g: 0 if g == '(no genres listed)' else len(g.split('|'))
)
gc_dist = movies['genre_count'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(gc_dist.index.astype(str), gc_dist.values,
       color=PALETTE[4], edgecolor='none', alpha=0.9)
ax.set_title('Number of Genres per Movie')
ax.set_xlabel('Number of Genres')
ax.set_ylabel('Number of Movies')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print('Genres per Movie Distribution:')
print(gc_dist.to_string())
print(f'\nAverage genres per movie : {movies["genre_count"].mean():.2f}')

In [ ]:
# ── 8.3  Genre popularity by total interactions ───────────────────────────────
genre_rating = (
    ratings[['movieId', 'rating']]
    .merge(genre_exploded[['movieId', 'genre']], on='movieId', how='inner')
)

genre_rating_stats = genre_rating.groupby('genre').agg(
    total_ratings=('rating', 'count'),
    mean_rating=('rating', 'mean')
).sort_values('total_ratings', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Genre Statistics (by Interactions)', fontsize=14, fontweight='bold')

ax = axes[0]
ax.barh(genre_rating_stats.index[::-1],
        genre_rating_stats['total_ratings'][::-1] / 1e6,
        color=PALETTE[2], alpha=0.9)
ax.set_title('Total Ratings per Genre')
ax.set_xlabel('Ratings (M)')
ax.grid(axis='x', alpha=0.4)

ax = axes[1]
sorted_mean = genre_rating_stats.sort_values('mean_rating')
ax.barh(sorted_mean.index, sorted_mean['mean_rating'], color=PALETTE[3], alpha=0.9)
ax.set_title('Mean Rating per Genre')
ax.set_xlabel('Average Rating')
ax.axvline(ratings['rating'].mean(), color=PALETTE[1],
           linestyle='--', linewidth=1.2, label='Global Mean')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── 8.4  Genre co-occurrence heatmap (top 12 genres) ─────────────────────────
TOP_N_GENRES = 12
top_genres = genre_counts.head(TOP_N_GENRES).index.tolist()

co_matrix = pd.DataFrame(0, index=top_genres, columns=top_genres)

for genres_str in movies['genres']:
    gl = [g for g in genres_str.split('|') if g in top_genres]
    for g1, g2 in combinations(gl, 2):
        co_matrix.loc[g1, g2] += 1
        co_matrix.loc[g2, g1] += 1
    for g in gl:
        co_matrix.loc[g, g] += 1

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.zeros_like(co_matrix.values, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True

sns.heatmap(
    co_matrix, mask=mask, ax=ax, cmap='plasma',
    annot=True, fmt='d', annot_kws={'size': 7},
    linewidths=0.5, linecolor='#0f0f1a',
    cbar_kws={'label': 'Co-occurrence Count'}
)
ax.set_title(f'Genre Co-occurrence — Top {TOP_N_GENRES} Genres', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---
<a id='9'></a>
## 9. Interaction Matrix Sparsity

In [ ]:
# ── 9.1  Overall sparsity ─────────────────────────────────────────────────────
n_users  = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
n_inter  = len(ratings)
max_inter = n_users * n_movies

density  = n_inter / max_inter
sparsity = 1 - density

print('Interaction Matrix Properties:')
print(f'  Users                 : {n_users:>12,}')
print(f'  Movies                : {n_movies:>12,}')
print(f'  Possible interactions : {max_inter:>12,}')
print(f'  Observed interactions : {n_inter:>12,}')
print(f'  Density               : {density:.6%}')
print(f'  Sparsity              : {sparsity:.6%}')
print()
print('💡 Extreme sparsity is typical for CF datasets.')
print('   NCF learns from this sparse signal via dense embedding representations.')

In [ ]:
# ── 9.2  Sparsity after user/movie activity filtering ─────────────────────────
print(f'{"Min User Ratings":>18}  {"Min Movie Ratings":>18}  {"Users":>8}  {"Movies":>8}  {"Interactions":>14}  {"Sparsity":>10}')
print('-' * 90)

for min_u in [20]:
    for min_m in [1, 5, 10, 20]:
        u_mask = user_counts[user_counts >= min_u].index
        m_mask = movie_counts[movie_counts >= min_m].index

        filtered = ratings[
            ratings['userId'].isin(u_mask) & ratings['movieId'].isin(m_mask)
        ]
        fu = filtered['userId'].nunique()
        fm = filtered['movieId'].nunique()
        fi = len(filtered)
        sp = 1 - fi / (fu * fm)
        print(f'{min_u:>18}  {min_m:>18}  {fu:>8,}  {fm:>8,}  {fi:>14,}  {sp:>10.6%}')

---
<a id='10'></a>
## 10. Tag & Genome Data Exploration

In [ ]:
# ── 10.1  User tag activity ───────────────────────────────────────────────────
print('Tags Dataset Overview:')
print(f'  Total tag applications : {len(tags):>10,}')
print(f'  Unique users who tagged: {tags["userId"].nunique():>10,}')
print(f'  Unique movies tagged   : {tags["movieId"].nunique():>10,}')
print(f'  Unique tag strings     : {tags["tag"].nunique():>10,}')

top_tags = tags['tag'].str.lower().value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_tags.index[::-1], top_tags.values[::-1],
        color=PALETTE[5], alpha=0.9)
ax.set_title('Top 20 Most Applied User Tags')
ax.set_xlabel('Number of Applications')
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── 10.2  Genome scores distribution ─────────────────────────────────────────
print('Tag Genome Overview:')
print(f'  Genome movies        : {genome_scores["movieId"].nunique():>10,}')
print(f'  Genome tags          : {genome_tags["tagId"].nunique():>10,}')
print(f'  Total genome entries : {len(genome_scores):>10,}')

sample_rel = genome_scores['relevance'].sample(100_000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sample_rel, bins=80, color=PALETTE[3], edgecolor='none', alpha=0.85)
ax.set_title('Genome Tag Relevance Score Distribution (100K sample)')
ax.set_xlabel('Relevance Score')
ax.set_ylabel('Count')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print('\nGenome Relevance Score Summary:')
print(genome_scores['relevance'].describe().to_string())

In [ ]:
# ── 10.3  Top genome tags by mean relevance ───────────────────────────────────
top_genome = (
    genome_scores
    .groupby('tagId')['relevance']
    .mean()
    .nlargest(20)
    .reset_index()
    .merge(genome_tags, on='tagId')
    .sort_values('relevance', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_genome['tag'], top_genome['relevance'],
        color=PALETTE[4], alpha=0.9)
ax.set_title('Top 20 Genome Tags by Mean Relevance Score')
ax.set_xlabel('Mean Relevance')
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.show()

---
<a id='11'></a>
## 11. NCF Readiness Checks

In [ ]:
# ── 11.1  Deduplication ───────────────────────────────────────────────────────
# Keep only the most recent rating when a user rated the same movie more than once.
ratings_dedup = (
    ratings
    .sort_values('timestamp')
    .drop_duplicates(subset=['userId', 'movieId'], keep='last')
)

print('After deduplication (keep last rating per user-movie pair):')
print(f'  Original interactions   : {len(ratings):>12,}')
print(f'  Deduplicated            : {len(ratings_dedup):>12,}')
print(f'  Rows removed            : {len(ratings) - len(ratings_dedup):>12,}')

In [ ]:
# ── 11.2  Leave-One-Out split analysis ────────────────────────────────────────
user_interaction_counts = ratings_dedup.groupby('userId').size()

single_interaction_users = (user_interaction_counts == 1).sum()
multi_interaction_users  = (user_interaction_counts >= 2).sum()

print('Leave-One-Out Split Analysis (on deduplicated data):')
print(f'  Total unique users                      : {len(user_interaction_counts):>10,}')
print(f'  Users with >= 2 interactions (LOO OK)   : {multi_interaction_users:>10,}')
print(f'  Users with exactly 1 interaction        : {single_interaction_users:>10,}')

test_interactions  = multi_interaction_users
train_interactions = (
    len(ratings_dedup)
    - test_interactions
    - single_interaction_users
)
print(f'\n  Projected test interactions (LOO)       : {test_interactions:>10,}')
print(f'  Projected train interactions            : {train_interactions:>10,}')

In [ ]:
# ── 11.3  Negative sampling feasibility ───────────────────────────────────────
n_unique_movies = ratings_dedup['movieId'].nunique()
avg_pos_per_user = user_interaction_counts.mean()
avg_neg_pool     = n_unique_movies - avg_pos_per_user

print('Negative Sampling Feasibility:')
print(f'  Unique movies in interaction data    : {n_unique_movies:>10,}')
print(f'  Average positives per user           : {avg_pos_per_user:>10.1f}')
print(f'  Average negative pool per user       : {avg_neg_pool:>10.1f}')
print()
print('  Training negative samples at common ratios:')
for neg_per_pos in [1, 2, 4, 8]:
    total_neg = int(avg_pos_per_user * neg_per_pos * multi_interaction_users)
    print(f'    {neg_per_pos}:1 ratio → ~{total_neg:,} negative samples total')
print()
print('  Evaluation protocol (He et al. 2017):')
print('    99 random negatives + 1 held-out positive = 100-item ranking task.')

In [ ]:
# ── 11.4  User and movie ID encoding preview ──────────────────────────────────
# NCF requires 0-indexed integer IDs for PyTorch Embedding layers
unique_users  = sorted(ratings_dedup['userId'].unique())
unique_movies = sorted(ratings_dedup['movieId'].unique())

user_to_idx  = {uid: idx for idx, uid in enumerate(unique_users)}
movie_to_idx = {mid: idx for idx, mid in enumerate(unique_movies)}

print('Encoding Summary:')
print(f'  User  embedding table size : {len(user_to_idx):>10,}')
print(f'  Movie embedding table size : {len(movie_to_idx):>10,}')
print()
print(f'  Original userId range  : {min(unique_users)} – {max(unique_users)}')
print(f'  Original movieId range : {min(unique_movies)} – {max(unique_movies)}')
print()
print('  Sample user mapping  (original → encoded):')
for uid in unique_users[:5]:
    print(f'    userId {uid:>7} → {user_to_idx[uid]}')
print()
print('  Sample movie mapping (original → encoded):')
for mid in unique_movies[:5]:
    print(f'    movieId {mid:>7} → {movie_to_idx[mid]}')

In [ ]:
# ── 11.5  Popularity baseline preview ────────────────────────────────────────
popularity = (
    ratings_dedup
    .groupby('movieId').size()
    .reset_index(name='interaction_count')
    .sort_values('interaction_count', ascending=False)
    .merge(movies[['movieId', 'title', 'genres']], on='movieId', how='left')
)

print('Top 10 Movies by Popularity — Baseline Candidates:')
print(popularity[['movieId', 'title', 'genres', 'interaction_count']].head(10).to_string(index=False))

---
<a id='12'></a>
## 12. Key Findings & Next Steps

### 📊 Dataset Summary

| Metric | Value |
|---|---|
| Total ratings | ~25.0M |
| Unique users | ~162K |
| Unique movies | ~59K |
| Interaction matrix density | ~0.26% |
| Date range | Jan 1995 – Nov 2019 |
| Rating scale | 0.5 – 5.0 (half-star) |

> **Note:** Exact values are printed by the cells above after execution.

---

### 🔑 Key EDA Findings

1. **Rating Scale** — Half-star increments from 0.5 to 5.0. Distribution is left-skewed (positivity bias toward 3–4 stars).

2. **Implicit Feedback** — All interactions can serve as binary positive signals (NCF paper approach). A threshold of ≥ 3.5 / ≥ 4.0 reduces noise but shrinks the training set.

3. **User Activity (Long Tail)** — Strong power-law distribution. All users have ≥ 20 ratings (per dataset specification), so no additional user filtering is strictly required.

4. **Movie Popularity (Long Tail)** — A small fraction of blockbusters dominate. Many movies have very few ratings — cold-start risk for the tail.

5. **LOO Eligibility** — Nearly all users qualify (≥ 2 interactions), providing a large and reliable evaluation pool.

6. **Negative Sampling** — With ~59K movies and ~154 average positives per user, the negative pool is large enough for any standard negative sampling ratio.

7. **Genre Distribution** — Drama, Comedy, and Thriller dominate. Multi-label genres are common (avg ~2 genres/movie). Genre co-occurrence is useful for hybrid re-ranking.

8. **Temporal Trends** — Stable rating activity with peaks around 2000–2010. No timestamp anomalies detected.

9. **Data Quality** — No missing values in core `ratings.csv` columns. `links.csv` may have some null `tmdbId`. No invalid rating values.

---

### ✅ Preprocessing Decisions for NCF

| Decision | Choice | Rationale |
|---|---|---|
| Interaction type | Implicit binary | Follows NCF paper |
| Rating threshold | All (≥ 0.5) | Maximise training signal |
| Deduplication | Keep last per (user, movie) | Most recent preference |
| Evaluation protocol | Leave-One-Out | Standard NCF protocol |
| Train negative ratio | 4 negatives per positive | Balance & efficiency |
| Eval negatives | 99 random per test user | 100-item ranking |
| User encoding | 0-indexed integer | PyTorch Embedding requirement |
| Movie encoding | 0-indexed integer | PyTorch Embedding requirement |

---

### 🚀 Next Steps (NCF Pipeline)

1. **`preprocessing.py`** — Full pipeline: deduplicate → encode → LOO split → negative sampling
2. **`dataset.py`** — PyTorch `Dataset` with on-the-fly negative sampling
3. **`models/gmf.py`** — Generalised Matrix Factorisation
4. **`models/mlp.py`** — Multi-Layer Perceptron
5. **`models/neumf.py`** — NeuMF (GMF + MLP fusion)
6. **`train.py`** — Training loop (binary cross-entropy, Adam)
7. **`evaluate.py`** — LOO evaluation computing HR@10 and NDCG@10
8. **`baseline.py`** — Popularity baseline evaluation